# Задача 3. Создание хеш-таблицы

Реализуем собственную хеш-таблицу для хранения пар **ключ → значение**.

Поддерживаем:

- вставку;
- поиск;
- удаление;
- обработку коллизий;
- автоматическое увеличение таблицы;
- автоматическое уменьшение таблицы;
- повторное использование удаленных ячеек;
- обновление существующего ключа;
- проверку наличия ключа;
- получение размера таблицы;
- статистику коллизий.

Для разрешения коллизий используется **открытая адресация с двойным хешированием**.

## Идея решения

Хеш-таблица хранит элементы в массиве.

Для каждого ключа вычисляется `hash(key)`, после чего по нему определяется стартовая позиция в массиве.

Если нужная ячейка свободна — кладем элемент туда.

Если ячейка занята другим ключом — возникает **коллизия**.  
Чтобы найти другое место, используем **двойное хеширование**: второй хеш задает шаг, с которым двигаемся по таблице.

При удалении мы не ставим `None`, а оставляем специальную метку `DELETED`.  
Это важно, потому что иначе можно случайно оборвать цепочку поиска после коллизии.

Если таблица становится слишком заполненной, увеличиваем ее размер и заново распределяем элементы.
Если после удалений таблица становится слишком пустой — уменьшаем.

In [2]:
class Entry:
    def __init__(self, key, value, hash_value):
        # ключ
        self.key = key

        # значение
        self.value = value

        # сохраняем hash, чтобы не считать его заново
        self.hash_value = hash_value


class HashTable:
    # специальная метка удаленной ячейки
    DELETED = object()

    def __init__(self, initial_capacity=11, max_load=0.70, min_load=0.20):
        # вместимость таблицы делаем простым числом
        self.initial_capacity = self._next_prime(max(3, initial_capacity))
        self.capacity = self.initial_capacity

        # сам массив, в котором хранятся элементы
        self.table = [None] * self.capacity

        # количество реальных элементов
        self.size = 0

        # количество удаленных ячеек DELETED
        self.deleted = 0

        # границы заполненности
        self.max_load = max_load
        self.min_load = min_load

        # статистика
        self.collisions = 0
        self.max_probe = 0

    # -------------------------------------------------
    # Работа с размером таблицы
    # -------------------------------------------------

    @staticmethod
    def _is_prime(number):
        # проверяем, является ли число простым
        if number < 2:
            return False

        if number % 2 == 0:
            return number == 2

        divisor = 3

        while divisor * divisor <= number:
            if number % divisor == 0:
                return False
            divisor += 2

        return True

    @classmethod
    def _next_prime(cls, number):
        # ищем ближайшее простое число >= number
        while not cls._is_prime(number):
            number += 1
        return number

    # -------------------------------------------------
    # Двойное хеширование
    # -------------------------------------------------

    def _probe_indexes(self, hash_value):
        # стартовая ячейка
        start = hash_value % self.capacity

        # шаг для разрешения коллизии
        step = 1 + (hash_value % (self.capacity - 1))

        # пробуем разные позиции таблицы
        for attempt in range(self.capacity):
            index = (start + attempt * step) % self.capacity
            yield index

    # -------------------------------------------------
    # Поиск ячейки
    # -------------------------------------------------

    def _find_slot(self, key, hash_value):
        # если встретим удаленную ячейку,
        # запомним первую такую позицию
        first_deleted = None

        probes = 0

        for index in self._probe_indexes(hash_value):
            probes += 1
            slot = self.table[index]

            # None означает, что дальше искать этот ключ не нужно
            if slot is None:
                if first_deleted is not None:
                    return first_deleted, False, probes
                return index, False, probes

            # DELETED не останавливает поиск
            if slot is self.DELETED:
                if first_deleted is None:
                    first_deleted = index
                continue

            # нашли именно нужный ключ
            if slot.hash_value == hash_value and slot.key == key:
                return index, True, probes

        # если свободного места не нашли,
        # но была удаленная ячейка — используем ее
        if first_deleted is not None:
            return first_deleted, False, probes

        return None, False, probes

    def _record_probe(self, probes):
        self.max_probe = max(self.max_probe, probes)

        # каждая дополнительная попытка после первой —
        # следствие коллизии
        if probes > 1:
            self.collisions += probes - 1

    # -------------------------------------------------
    # Resize / rehash
    # -------------------------------------------------

    def _insert_without_resize(self, key, value, hash_value):
        index, found, _ = self._find_slot(key, hash_value)

        if found:
            self.table[index].value = value
            return

        self.table[index] = Entry(key, value, hash_value)
        self.size += 1

    def _resize(self, new_capacity):
        # новая вместимость тоже должна быть простым числом
        new_capacity = self._next_prime(
            max(self.initial_capacity, new_capacity)
        )

        # сохраняем только живые элементы
        old_entries = []

        for slot in self.table:
            if slot is not None and slot is not self.DELETED:
                old_entries.append(slot)

        # создаем новую пустую таблицу
        self.capacity = new_capacity
        self.table = [None] * self.capacity
        self.size = 0
        self.deleted = 0

        # заново распределяем все элементы
        for entry in old_entries:
            self._insert_without_resize(
                entry.key,
                entry.value,
                entry.hash_value
            )

    # -------------------------------------------------
    # INSERT
    # -------------------------------------------------

    def put(self, key, value):
        hash_value = hash(key)

        index, found, probes = self._find_slot(key, hash_value)

        # если ключ уже существует — просто обновляем значение
        if found:
            self.table[index].value = value
            self._record_probe(probes)
            return

        # учитываем и живые, и удаленные ячейки,
        # потому что обе занимают место в массиве
        occupied = self.size + self.deleted

        # если таблица стала слишком заполненной — увеличиваем
        if (occupied + 1) / self.capacity > self.max_load:
            self._resize(self.capacity * 2)

            index, found, probes = self._find_slot(key, hash_value)

        self._record_probe(probes)

        # если переиспользуем удаленную ячейку
        if self.table[index] is self.DELETED:
            self.deleted -= 1

        self.table[index] = Entry(key, value, hash_value)
        self.size += 1

    # -------------------------------------------------
    # SEARCH
    # -------------------------------------------------

    def get(self, key, default=None):
        hash_value = hash(key)

        index, found, probes = self._find_slot(key, hash_value)
        self._record_probe(probes)

        if found:
            return self.table[index].value

        return default

    # -------------------------------------------------
    # DELETE
    # -------------------------------------------------

    def delete(self, key):
        hash_value = hash(key)

        index, found, probes = self._find_slot(key, hash_value)
        self._record_probe(probes)

        if not found:
            raise KeyError(key)

        value = self.table[index].value

        # вместо None оставляем специальную метку DELETED
        self.table[index] = self.DELETED
        self.size -= 1
        self.deleted += 1

        # если таблица полностью опустела — возвращаем минимальный размер
        if self.size == 0 and self.capacity != self.initial_capacity:
            self._resize(self.initial_capacity)

        # если таблица стала слишком пустой — уменьшаем
        elif (
            self.capacity > self.initial_capacity
            and self.size / self.capacity < self.min_load
        ):
            self._resize(self.capacity // 2)

        # если накопилось слишком много DELETED,
        # пересобираем таблицу того же размера
        elif self.deleted > self.size and self.deleted > 5:
            self._resize(self.capacity)

        return value

    # -------------------------------------------------
    # Дополнительные возможности
    # -------------------------------------------------

    def __len__(self):
        return self.size

    def __contains__(self, key):
        hash_value = hash(key)

        _, found, probes = self._find_slot(key, hash_value)
        self._record_probe(probes)

        return found

    def __setitem__(self, key, value):
        # позволяет писать: table[key] = value
        self.put(key, value)

    def __getitem__(self, key):
        # позволяет писать: table[key]
        hash_value = hash(key)

        index, found, probes = self._find_slot(key, hash_value)
        self._record_probe(probes)

        if not found:
            raise KeyError(key)

        return self.table[index].value

    def __delitem__(self, key):
        # позволяет писать: del table[key]
        self.delete(key)

    @property
    def load_factor(self):
        return self.size / self.capacity

    def items(self):
        for slot in self.table:
            if slot is not None and slot is not self.DELETED:
                yield slot.key, slot.value

    def stats(self):
        return {
            "size": self.size,
            "capacity": self.capacity,
            "load_factor": round(self.load_factor, 3),
            "deleted_slots": self.deleted,
            "collisions": self.collisions,
            "max_probe": self.max_probe,
        }

    def debug_print(self):
        # показывает внутреннее устройство таблицы
        for index, slot in enumerate(self.table):
            if slot is None:
                state = "EMPTY"
            elif slot is self.DELETED:
                state = "DELETED"
            else:
                state = f"{slot.key!r}: {slot.value!r}"

            print(f"{index:>3}: {state}")

    def __repr__(self):
        content = ", ".join(
            f"{key!r}: {value!r}"
            for key, value in self.items()
        )
        return "{" + content + "}"

## Базовое использование

Хеш-таблица умеет работать и через методы `put/get/delete`, и почти как обычный Python-словарь.

In [3]:
table = HashTable()

table.put("name", "Настя")
table.put("age", 23)
table.put("city", "Москва")

print(table)

table["job"] = "analyst"

print("job:", table["job"])
print("name:", table.get("name"))
print("unknown:", table.get("unknown"))

table.delete("age")

print("После удаления age:", table)
print("name" in table)
print("age" in table)
print("Размер:", len(table))

{'city': 'Москва', 'age': 23, 'name': 'Настя'}
job: analyst
name: Настя
unknown: None
После удаления age: {'city': 'Москва', 'job': 'analyst', 'name': 'Настя'}
True
False
Размер: 3


## Коллизии

Коллизия возникает, когда разные ключи хотят попасть в одну и ту же ячейку.

Чтобы специально показать, что наша таблица умеет с этим работать, создадим ключи, которые всегда возвращают один и тот же `hash`.

In [4]:
class CollisionKey:
    def __init__(self, name):
        self.name = name

    def __hash__(self):
        # специально создаем одинаковый hash
        return 1

    def __eq__(self, other):
        return (
            isinstance(other, CollisionKey)
            and self.name == other.name
        )

    def __repr__(self):
        return f"Key({self.name})"

In [5]:
collision_table = HashTable(initial_capacity=7)

key_a = CollisionKey("A")
key_b = CollisionKey("B")
key_c = CollisionKey("C")

collision_table[key_a] = 100
collision_table[key_b] = 200
collision_table[key_c] = 300

collision_table.debug_print()

  0: EMPTY
  1: Key(A): 100
  2: EMPTY
  3: Key(B): 200
  4: EMPTY
  5: Key(C): 300
  6: EMPTY


## Как разрешаются коллизии

Используется **двойное хеширование**.

Первая формула задает стартовую позицию:

`start = hash(key) % capacity`

Вторая задает шаг:

`step = 1 + hash(key) % (capacity - 1)`

Если стартовая ячейка занята, алгоритм идет по позициям:

`start`

`start + step`

`start + 2 × step`

`start + 3 × step`

и так далее, пока не найдет нужный ключ или свободную ячейку.

Размер таблицы делается простым числом, чтобы такой шаг хорошо обходил ячейки таблицы и не зацикливался на небольшом наборе позиций.

## Почему при удалении нужен DELETED

Если после коллизии ключ оказался не в стартовой ячейке, поиск может пройти через несколько позиций.

Если при удалении просто поставить `None`, поиск может преждевременно остановиться и решить, что нужного ключа дальше нет.

Поэтому используется специальная отметка `DELETED`.

Она означает:

> здесь раньше был элемент, поэтому поиск нужно продолжить.

При этом новая вставка может повторно использовать такую ячейку.

In [6]:
print("До удаления:")
collision_table.debug_print()

collision_table.delete(key_b)

print("\nПосле удаления B:")
collision_table.debug_print()

key_d = CollisionKey("D")
collision_table[key_d] = 400

print("\nПосле вставки D:")
collision_table.debug_print()

До удаления:
  0: EMPTY
  1: Key(A): 100
  2: EMPTY
  3: Key(B): 200
  4: EMPTY
  5: Key(C): 300
  6: EMPTY

После удаления B:
  0: EMPTY
  1: Key(A): 100
  2: EMPTY
  3: DELETED
  4: EMPTY
  5: Key(C): 300
  6: EMPTY

После вставки D:
  0: EMPTY
  1: Key(A): 100
  2: EMPTY
  3: Key(D): 400
  4: EMPTY
  5: Key(C): 300
  6: EMPTY


## Автоматический resize

Если таблица заполнена слишком сильно, коллизий становится больше и поиск замедляется.

Поэтому при заполнении выше **70%** таблица автоматически увеличивается.

Если после удаления элементов она заполнена меньше чем на **20%**, таблица уменьшается.

После изменения размера все элементы заново распределяются по новой таблице.

In [7]:
resize_table = HashTable(initial_capacity=5)

print("Начальная capacity:", resize_table.capacity)

for number in range(20):
    resize_table[number] = number * 10

print("Capacity после вставок:", resize_table.capacity)
print("Количество элементов:", len(resize_table))
print("Load factor:", resize_table.load_factor)

Начальная capacity: 5
Capacity после вставок: 47
Количество элементов: 20
Load factor: 0.425531914893617


## Статистика

Дополнительно таблица умеет показывать:

- количество элементов;
- текущую вместимость;
- load factor;
- количество удаленных ячеек;
- число столкновений при поиске позиций;
- максимальную длину поиска подходящей ячейки.

In [8]:
print(collision_table.stats())

{'size': 3, 'capacity': 7, 'load_factor': 0.429, 'deleted_slots': 0, 'collisions': 7, 'max_probe': 4}


## Визуализация работы алгоритма

Представим таблицу вместимостью 7.

Допустим, три разных ключа получили одинаковый hash и хотят начать с одной позиции.

### Вставка A

Первая подходящая ячейка свободна:

`[EMPTY, A, EMPTY, EMPTY, EMPTY, EMPTY, EMPTY]`

### Вставка B

B хочет попасть туда же, где уже лежит A.

Возникает коллизия.

По второму хешу вычисляется шаг, и B перемещается в следующую допустимую позицию:

`[EMPTY, A, EMPTY, B, EMPTY, EMPTY, EMPTY]`

### Вставка C

C тоже начинает с позиции A.

Она занята.

Следующая позиция по шагу тоже занята B.

Поэтому C идет дальше:

`[EMPTY, A, EMPTY, B, EMPTY, C, EMPTY]`

### Удаление B

Мы не ставим `None`, а оставляем:

`[EMPTY, A, EMPTY, DELETED, EMPTY, C, EMPTY]`

Так поиск C понимает, что после удаленной ячейки нужно продолжать движение.

### Новая вставка D

Если D проходит через эту же цепочку, он может переиспользовать ячейку `DELETED`:

`[EMPTY, A, EMPTY, D, EMPTY, C, EMPTY]`

## Оценка сложности

### Вставка

В среднем: `O(1)`

По hash сразу вычисляется стартовая позиция. При небольшом количестве коллизий нужно проверить только несколько ячеек.

В худшем случае: `O(n)`

Такое возможно, если коллизий очень много и приходится пройти значительную часть таблицы.

### Поиск

В среднем: `O(1)`

В худшем случае: `O(n)`

### Удаление

В среднем: `O(1)`

В худшем случае: `O(n)`

### Resize

Перестроение таблицы занимает `O(n)`, потому что нужно заново разместить существующие элементы.

Но resize выполняется не при каждой вставке, поэтому **амортизированная** сложность вставки остается `O(1)`.

### Память

`O(n)`

Таблица хранит массив, размер которого пропорционален числу элементов.

## Тесты

In [9]:
import unittest
import sys


class TestHashTable(unittest.TestCase):

    def test_insert_and_get(self):
        table = HashTable()

        table.put("a", 10)
        table.put("b", 20)

        self.assertEqual(table.get("a"), 10)
        self.assertEqual(table.get("b"), 20)

    def test_update(self):
        table = HashTable()

        table.put("a", 10)
        table.put("a", 50)

        self.assertEqual(table.get("a"), 50)
        self.assertEqual(len(table), 1)

    def test_delete(self):
        table = HashTable()

        table["a"] = 10
        removed = table.delete("a")

        self.assertEqual(removed, 10)
        self.assertIsNone(table.get("a"))

    def test_delete_missing_key(self):
        table = HashTable()

        with self.assertRaises(KeyError):
            table.delete("unknown")

    def test_contains(self):
        table = HashTable()

        table["hello"] = 100

        self.assertTrue("hello" in table)
        self.assertFalse("world" in table)

    def test_python_syntax(self):
        table = HashTable()

        table["name"] = "Nastya"

        self.assertEqual(table["name"], "Nastya")

        del table["name"]

        self.assertFalse("name" in table)

    def test_collision(self):
        table = HashTable(initial_capacity=7)

        key1 = CollisionKey("A")
        key2 = CollisionKey("B")
        key3 = CollisionKey("C")

        table[key1] = 1
        table[key2] = 2
        table[key3] = 3

        self.assertEqual(table[key1], 1)
        self.assertEqual(table[key2], 2)
        self.assertEqual(table[key3], 3)

    def test_delete_with_collision(self):
        table = HashTable(initial_capacity=7)

        key1 = CollisionKey("A")
        key2 = CollisionKey("B")
        key3 = CollisionKey("C")

        table[key1] = 1
        table[key2] = 2
        table[key3] = 3

        table.delete(key2)

        # удаление B не должно сломать поиск C
        self.assertEqual(table[key3], 3)

    def test_resize(self):
        table = HashTable(initial_capacity=5)

        old_capacity = table.capacity

        for number in range(20):
            table[number] = number

        self.assertGreater(table.capacity, old_capacity)

        # после resize все значения должны сохраниться
        for number in range(20):
            self.assertEqual(table[number], number)

    def test_different_key_types(self):
        table = HashTable()

        table[1] = "integer"
        table["1"] = "string"
        table[(1, 2)] = "tuple"
        table[None] = "none"

        self.assertEqual(table[1], "integer")
        self.assertEqual(table["1"], "string")
        self.assertEqual(table[(1, 2)], "tuple")
        self.assertEqual(table[None], "none")


suite = unittest.TestLoader().loadTestsFromTestCase(TestHashTable)

runner = unittest.TextTestRunner(
    stream=sys.stdout,
    verbosity=2
)

result = runner.run(suite)

test_collision (__main__.TestHashTable) ... ok
test_contains (__main__.TestHashTable) ... ok
test_delete (__main__.TestHashTable) ... ok
test_delete_missing_key (__main__.TestHashTable) ... ok
test_delete_with_collision (__main__.TestHashTable) ... ok
test_different_key_types (__main__.TestHashTable) ... ok
test_insert_and_get (__main__.TestHashTable) ... ok
test_python_syntax (__main__.TestHashTable) ... ok
test_resize (__main__.TestHashTable) ... ok
test_update (__main__.TestHashTable) ... ok

----------------------------------------------------------------------
Ran 10 tests in 0.005s

OK
